In [1]:
# pip install pandas numpy matplotlib openpyxl

## DATASET OVERVIEW

In [4]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE_DIR = Path("..").resolve()          # app/
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = DATA_DIR / "outputs"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("RAW_DIR:", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("OUTPUTS_DIR:", OUTPUTS_DIR)

# Load data
_df_train= pd.read_csv(RAW_DIR/"algebra_2006_2007_train.txt", sep="\t")
_df_test = pd.read_csv(RAW_DIR/"algebra_2006_2007_test.txt", sep="\t")

print("Train shape:", _df_train.shape)
print("Test shape:", _df_test.shape)
print("Columns:", _df_train.columns.to_list())
print("Number of students:", _df_train["Anon Student Id"].nunique())
print("Number of problems:", _df_train["Problem Name"].nunique())
print("Number of steps:", _df_train["Step Name"].nunique())
print("Number of original KCs:", _df_train["KC(Default)"].nunique())


BASE_DIR: C:\Users\Melany Nuria\Desktop\TFG\adaptive_bayesian_its\backend_fastAPI\app
RAW_DIR: C:\Users\Melany Nuria\Desktop\TFG\adaptive_bayesian_its\backend_fastAPI\app\data\raw
PROCESSED_DIR: C:\Users\Melany Nuria\Desktop\TFG\adaptive_bayesian_its\backend_fastAPI\app\data\processed
OUTPUTS_DIR: C:\Users\Melany Nuria\Desktop\TFG\adaptive_bayesian_its\backend_fastAPI\app\data\outputs
Train shape: (2270384, 19)
Test shape: (19342, 19)
Columns: ['Row', 'Anon Student Id', 'Problem Hierarchy', 'Problem Name', 'Problem View', 'Step Name', 'Step Start Time', 'First Transaction Time', 'Correct Transaction Time', 'Step End Time', 'Step Duration (sec)', 'Correct Step Duration (sec)', 'Error Step Duration (sec)', 'Correct First Attempt', 'Incorrects', 'Hints', 'Corrects', 'KC(Default)', 'Opportunity(Default)']
Number of students: 1338
Number of problems: 91913
Number of steps: 316896
Number of original KCs: 1703


In [ ]:
cols_needed = [
    "Anon Student Id",
    "Problem Name",
    "Step Name",
    "KC(Default)",
    "Correct First Attempt",
    "Incorrects",
    "Hints",
    "Corrects",
    "Step Start Time",
    "First Transaction Time",
    "Correct Transaction Time",
    "Step End Time"
]


### Helpers

In [5]:
def convert_time_columns(df):
    df = df.copy()

    time_cols = [
        "Step Start Time",
        "First Transaction Time",
        "Correct Transaction Time",
        "Step End Time"
    ]

    for col in time_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    return df


def convert_numeric_columns(df):
    df = df.copy()

    numeric_cols = [
        "Correct First Attempt",
        "Incorrects",
        "Hints",
        "Corrects"
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


# =========================================================
# KC MAPPING
# =========================================================
def map_kc(kc):
    if pd.isna(kc):
        return None

    kc_lower = str(kc).lower()

    # Exclude patterns clearly not aligned with the selected domain
    exclude_patterns = [
        "skillrule: variable in denominator;",
        "clt-r-den",
        "clt-r-num",
        "combine-like-terms-r-sp",
    ]
    if any(p in kc_lower for p in exclude_patterns):
        return "Other"

    # 1. Expand / Eliminate Parentheses
    if (
        "rule: clt nested, parens" in kc_lower
        or "skillrule: eliminate parens;" in kc_lower
        or "skillrule: select eliminate parens;" in kc_lower
        or "skillrule: calculate eliminate parens;" in kc_lower
        or "skillrule: do eliminate parens - whole;" in kc_lower
        or "distribute" in kc_lower
        or "perform-mult-whole-sp" in kc_lower
    ):
        return "Expand / Eliminate Parentheses"

    # 2. Combine Like Terms
    if (
        "skillrule: combine like terms, no var;" in kc_lower
        or "skillrule: select combine terms;" in kc_lower
        or "skillrule: do combine terms - whole;" in kc_lower
        or "combineliketerms" in kc_lower
        or "combine-like-terms-sp" in kc_lower
        or "combine-like-terms-whole-sp" in kc_lower
        or (
            "clt" in kc_lower
            and "parens" not in kc_lower
            and "r-den" not in kc_lower
            and "r-num" not in kc_lower
        )
    ):
        return "Combine Like Terms"

    # 3. Move Constants Across the Equation
    if (
        "skillrule: remove constant;" in kc_lower
        or "skillrule: ax+b=c, negative;" in kc_lower
        or "skillrule: isolate positive;" in kc_lower
        or "skillrule: isolate negative;" in kc_lower
        or "move constant" in kc_lower
        or "[solveroperation add]" in kc_lower
        or "[solveroperation subtract]" in kc_lower
        or "skillrule: add/subtract;" in kc_lower
    ):
        return "Move Constants Across the Equation"

    # 4. Move Variable Terms to One Side
    if (
        "skillrule: consolidate vars, no coeff;" in kc_lower
        or "skillrule: consolidate vars with coeff;" in kc_lower
        or "skillrule: consolidate vars, any;" in kc_lower
        or "skillrule: extract to consolidate vars;" in kc_lower
    ):
        return "Move Variable Terms to One Side"

    # 5. Remove Coefficient
    if (
        "skillrule: remove coefficient;" in kc_lower
        or "skillrule: remove positive coefficient;" in kc_lower
        or "skillrule: remove negative coefficient;" in kc_lower
        or "[solveroperation multiply]" in kc_lower
        or "[solveroperation divide]" in kc_lower
        or "solveroperation mt" in kc_lower
        or "solveroperation rf" in kc_lower
        or "rf right" in kc_lower
        or "rf left" in kc_lower
        or "skillrule: multiply/divide;" in kc_lower
    ):
        return "Remove Coefficient"

    # 6. Normalize Negative Variable / Sign
    if (
        "skillrule: make variable positive;" in kc_lower
        or "skillrule: calculate negative coefficient;" in kc_lower
    ):
        return "Normalize Negative Variable / Sign"

    return "Other"


def preprocess_kc(
    df,
    remove_other=False,
    drop_original_kc_cols=False,
    filter_irrelevant=False
):
    df = df.copy()

    # Remove rows without original KC
    df = df.dropna(subset=["KC(Default)"])

    # Split multi-KC rows
    df["KC"] = df["KC(Default)"].astype(str).str.split("~~")

    # One row per KC
    df = df.explode("KC")
    input_rows = df.shape[0]

    # Clean KC values
    df["KC"] = df["KC"].astype(str).str.strip()
    df = df[df["KC"] != ""]
    df = df[df["KC"].str.lower() != "nan"]

    # Optional domain filtering
    if filter_irrelevant:
        irrelevant_patterns = [
            "action", "compare", "axis", "-row",
            "unspecified", "unknown", "any form", "entering a",
            "plot", "graph", "probability", "median", "mean",
            "mode", "rate", "ratio", "scientific notation", "slope",
            "square root", "hypotenuse", "geometry", "quadrat", "exponent"
        ]

        def is_relevant(kc):
            if pd.isna(kc):
                return False
            kc = str(kc).lower()
            return not any(pattern in kc for pattern in irrelevant_patterns)

        df = df[df["KC"].apply(is_relevant)]

    # Map to reduced KC taxonomy
    df["new_KC"] = df["KC"].apply(map_kc)

    if remove_other:
        df = df[df["new_KC"] != "Other"]

    if drop_original_kc_cols:
        cols_to_drop = [col for col in ["KC(Default)", "KC"] if col in df.columns]
        df = df.drop(columns=cols_to_drop)

    df = df.reset_index(drop=True)
    output_rows = df.shape[0]

    print(f"Dataset input rows: {input_rows}")
    print(f"Dataset output rows: {output_rows}\n")

    return df


## KC election

Some steps have multiple KCs associated with them; when this occurs, they are separated by ~~. In this section, we create one row per KC and then classify them to obtain a reduced and interpretable set of KCs related to first-degree linear equations:

1. Remove Constant – Moving constant terms from one side of the equation to the other.
2. Combine Like Terms – Simplifying expressions by combining terms of the same type.
3. Consolidate Variable Terms – Bringing all variable terms to one side of the equation.
4. Eliminate Parentheses / Distribute – Removing parentheses by applying the distributive property.
5. Multiply or Divide Terms – Performing multiplication or division to simplify expressions.
6. Remove Coefficient – Eliminating the coefficient of the variable to isolate it.
7. Handle Negative Variable – Correctly managing negative signs when isolating the variable.

In [6]:
# 1. KEEP ONLY RELEVANT COLUMNS
cols_needed = [
    "KC(Default)",
    "Problem Name",
    "Step Name",
    "Correct First Attempt",
    "Anon Student Id"
]

df = _df_train[cols_needed].copy()

df = preprocess_kc(df, remove_other=True, filter_irrelevant=True)

# 4. BUILD MAIN KC -> PROBLEM -> STEP TABLE
kc_problem_step = (
    df.groupby(["KC", "new_KC", "Problem Name", "Step Name"])
      .agg(
          appearances=("KC", "size"),
          students=("Anon Student Id", "nunique"),
          avg_correct=("Correct First Attempt", "mean")
      )
      .reset_index()
      .sort_values(["new_KC", "appearances"], ascending=[True, False])
)

# 5. UNIQUE KCs WITH COUNTS
kc_summary = (
    df.groupby("KC")
      .agg(
          total_appearances=("KC", "size"),
          n_problems=("Problem Name", "nunique"),
          n_steps=("Step Name", "nunique"),
          n_students=("Anon Student Id", "nunique"),
          avg_correct=("Correct First Attempt", "mean")
      )
      .reset_index()
      .sort_values("total_appearances", ascending=False)
)

# 6. EXPORT FOR MANUAL REVIEW
kc_problem_step.to_excel("kc_problem_step_mapping.xlsx", index=False)
kc_summary.to_excel("kc_summary.xlsx", index=False)


Dataset input rows: 1852340
Dataset output rows: 410074



ModuleNotFoundError: No module named 'openpyxl'

Final dataset with only relevant KC columns

In [ ]:
df_train = _df_train[cols_needed].copy()
df_test = _df_test[cols_needed].copy()

df_train = convert_time_columns(df_train)
df_test = convert_time_columns(df_test)

df_train = convert_numeric_columns(df_train)
df_test = convert_numeric_columns(df_test)

df_train = preprocess_kc(df_train,remove_other=True,drop_original_kc_cols=True,filter_irrelevant=True)
df_test = preprocess_kc(df_test,remove_other=True,drop_original_kc_cols=True,filter_irrelevant=True)

# Sort in true chronological order per student
df_train = df_train.sort_values(
    by=["Anon Student Id", "Step Start Time", "Problem Name", "Step Name"]
).reset_index(drop=True)

df_test = df_test.sort_values(
    by=["Anon Student Id", "Step Start Time", "Problem Name", "Step Name"]
).reset_index(drop=True)

# REMOVE DUPLICATES
# Include Step Start Time to preserve separate interactions
subset_cols = [
    "Anon Student Id",
    "Problem Name",
    "Step Name",
    "new_KC",
    "Correct First Attempt",
    "Step Start Time"
]

duplicates_train = df_train.duplicated(subset=subset_cols)
duplicates_test = df_test.duplicated(subset=subset_cols)

n_duplicates_train = duplicates_train.sum()
n_duplicates_test = duplicates_test.sum()

percentage_train = (n_duplicates_train / len(df_train)) * 100 if len(df_train) > 0 else 0
percentage_test = (n_duplicates_test / len(df_test)) * 100 if len(df_test) > 0 else 0

print(f"Train duplicated interactions: {n_duplicates_train}")
print(f"Train duplicate percentage: {percentage_train:.2f}%")

print(f"Test duplicated interactions: {n_duplicates_test}")
print(f"Test duplicate percentage: {percentage_test:.2f}%")

df_train = df_train.drop_duplicates(subset=subset_cols).reset_index(drop=True)
df_test = df_test.drop_duplicates(subset=subset_cols).reset_index(drop=True)

print("Remaining train duplicates:", df_train.duplicated(subset=subset_cols).sum())
print("Remaining test duplicates:", df_test.duplicated(subset=subset_cols).sum())

# =========================================================
# FINAL DATASET INFO
# =========================================================
print("\nFINAL TRAIN DATASET")
print("Shape:", df_train.shape)
print("Columns:", df_train.columns.to_list())
print("Number of students:", df_train["Anon Student Id"].nunique())
print("Number of problems:", df_train["Problem Name"].nunique())
print("Number of steps:", df_train["Step Name"].nunique())
print("Number of reduced KCs:", df_train["new_KC"].nunique())

print("\nReduced KC counts:")
print(df_train["new_KC"].value_counts())


Train shape: (410074, 19)
Columns: ['Row', 'Anon Student Id', 'Problem Hierarchy', 'Problem Name', 'Problem View', 'Step Name', 'Step Start Time', 'First Transaction Time', 'Correct Transaction Time', 'Step End Time', 'Step Duration (sec)', 'Correct Step Duration (sec)', 'Error Step Duration (sec)', 'Correct First Attempt', 'Incorrects', 'Hints', 'Corrects', 'Opportunity(Default)', 'new_KC']
Number of students: 1165
Number of problems: 80054
Number of steps: 236087
Number of KCs: 6
new_KC
Remove Coefficient                    118975
Combine Like Terms                    118964
Move Constants Across the Equation    105059
Expand / Eliminate Parentheses         59619
Move Variable Terms to One Side         6522
Normalize Negative Variable / Sign       935
Name: count, dtype: int64


: 

## Final cleaned files

In [ ]:
train_file = PROCESSED_DIR / "df_train_kc_cleaned.csv"
test_file = PROCESSED_DIR / "df_test_kc_cleaned.csv"

df_train.to_csv(train_file, index=False)
df_test.to_csv(test_file, index=False)

print(f"Saved: {train_file}")
print(f"Saved: {test_file}")



Duplicated interactions: 3681
Percentage: 0.90%
Remaining duplicates: 0


: 

Notas: 
- cata studiante tiene su P(L) prob. of learning 
- este dataset nos ayuda a obtener las probabilidades Pl0, pt, pg, ps para cada KC 
- despues de cada ejercicio realizadp la P(L) del estudiante se actualiza

# mirar si cambiar a 6 kc!!! 
cambiar consolidate? 
